# Road Damage Detection - YOLOv8s, YOLO11n and experiments
Training and evaluation notebook (Google Colab, Tesla T4). YOLO11s and the data checks are in `yolo11s.ipynb`.

**Expected Drive layout:** `MyDrive/road_damage_runs/road-damage.v1i.yolov8.zip`. Every run saves its results to `MyDrive/road_damage_runs/<run_name>/`.

## 1. Setup

In [1]:
!pip install -q ultralytics

In [2]:
import os, time, glob, shutil, zipfile, yaml
import numpy as np
from PIL import Image
import torch
from ultralytics import YOLO
from google.colab import drive

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/road_damage_runs"
os.makedirs(SAVE_DIR, exist_ok=True)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Mounted at /content/drive


## 2. Load data and remove train/val/test leakage

In [3]:
data_dir = "/content/road-damage"
shutil.rmtree(data_dir, ignore_errors=True)
zips = glob.glob(f"{SAVE_DIR}/*.zip")
print(zips)
with zipfile.ZipFile(zips[0]) as z:
    z.extractall(data_dir)

yamls = [p for p in glob.glob(f"{data_dir}/**/data.yaml", recursive=True) if {"train", "valid", "test"} <= set(os.listdir(os.path.dirname(p)))]
root = os.path.dirname(yamls[0])
d = yaml.safe_load(open(yamls[0]))
names = d["names"]

# Original image name without the Roboflow hash
def base(p):
    return os.path.basename(p).split(".rf.")[0]

files = {s: glob.glob(f"{root}/{s}/images/*") for s in ["train", "valid", "test"]}
test_names = {base(p) for p in files["test"]}
valid_names = {base(p) for p in files["valid"]}

# Remove leaked images from train and valid, test stays untouched
train_clean = [p for p in files["train"] if base(p) not in test_names | valid_names]
valid_clean = [p for p in files["valid"] if base(p) not in test_names]
open("/content/train.txt", "w").write("\n".join(train_clean))
open("/content/valid.txt", "w").write("\n".join(valid_clean))

d["path"] = root
d["train"], d["val"], d["test"] = "/content/train.txt", "/content/valid.txt", f"{root}/test/images"
DATA_YAML = "/content/data.yaml"
yaml.dump(d, open(DATA_YAML, "w"))
print(len(train_clean), len(valid_clean), len(files["test"]))

['/content/drive/MyDrive/road_damage_runs/road-damage.v1i.yolov8.zip']
4591 1123 681


## 3. Settings (identical for all models)

In [4]:
print(torch.__version__, torch.cuda.is_available())

SEED = 42
IMGSZ = 640
BATCH = 16
EPOCHS = 40
PATIENCE = 15

2.11.0+cu130 True


## 4. Clean test set
Test images that are near-identical to training images (perceptual hash) are removed, leaving 592 test images.

In [5]:
def ahash(p, s=16):
    a = np.asarray(Image.open(p).convert("L").resize((s, s)), dtype=np.float32)
    return (a > a.mean()).flatten()

tr = np.array([ahash(p) for p in train_clean])
test_clean = [p for p in files["test"] if int((tr != ahash(p)).sum(1).min()) > 10]
open("/content/test_clean.txt", "w").write("\n".join(test_clean))
d2 = dict(d); d2["test"] = "/content/test_clean.txt"
yaml.dump(d2, open("/content/data_clean.yaml", "w"))
print(len(test_clean))

592


## 5. Model 1: YOLOv8s (40 epochs)
`map6` = mean mAP over 6 classes, excluding `edge` (only 19 training examples).

In [6]:
WEIGHTS, NAME = "yolov8s.pt", "yolov8s_run"

YOLO(WEIGHTS).train(data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED,
                    patience=PATIENCE, device=0, project=SAVE_DIR, name=NAME, exist_ok=True)

best_path = f"{SAVE_DIR}/{NAME}/weights/best.pt"
best = YOLO(best_path)

def map6(m):
    idx = [i for i, c in enumerate(m.box.ap_class_index) if c != 3]
    return round(float(m.box.all_ap[idx, 0].mean()), 3), round(float(m.box.all_ap[idx].mean()), 3)

mv = best.val(data=DATA_YAML, split="val", imgsz=IMGSZ, batch=BATCH, device=0, plots=False)
mt = best.val(data="/content/data_clean.yaml", split="test", imgsz=IMGSZ, batch=BATCH, device=0, plots=True)

print("RESULT", NAME)
print("val  mAP50, mAP50-95:", map6(mv))
print("test mAP50, mAP50-95:", map6(mt))
print("test precision, recall:", round(mt.box.mp, 3), round(mt.box.mr, 3))
print("inference ms:", round(mt.speed["inference"], 1), "| size MB:", round(os.path.getsize(best_path) / 1e6, 1))

Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_run, nbs=64, nms=None, opset=None, optimize=False, optimizer=au

## 6. Model 2: YOLO11n (40 epochs)

In [6]:
def map6(m):
    idx = [i for i, c in enumerate(m.box.ap_class_index) if c != 3]
    return round(float(m.box.all_ap[idx, 0].mean()), 3), round(float(m.box.all_ap[idx].mean()), 3)

RUNS = [("yolo11n.pt", "yolo11n_run")]

for WEIGHTS, NAME in RUNS:
    YOLO(WEIGHTS).train(data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED,
                        patience=PATIENCE, device=0, project=SAVE_DIR, name=NAME, exist_ok=True)
    best_path = f"{SAVE_DIR}/{NAME}/weights/best.pt"
    best = YOLO(best_path)
    mv = best.val(data=DATA_YAML, split="val", imgsz=IMGSZ, batch=BATCH, device=0, plots=False)
    mt = best.val(data="/content/data_clean.yaml", split="test", imgsz=IMGSZ, batch=BATCH, device=0, plots=True)
    print("RESULT", NAME)
    print("val  mAP50, mAP50-95:", map6(mv))
    print("test mAP50, mAP50-95:", map6(mt))
    print("test precision, recall:", round(mt.box.mp, 3), round(mt.box.mr, 3))
    print("inference ms:", round(mt.speed["inference"], 1), "| size MB:", round(os.path.getsize(best_path) / 1e6, 1))

Ultralytics 8.4.194 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_run, nbs=64, nms=None, opset=None, optimize=False, optimizer=au

## 7. Hyperparameter tuning (short runs on YOLO11n)
15 epochs on 30% of the data. Note: in short runs Ultralytics picks the optimizer automatically and ignores `lr0`, so the learning-rate comparison is not valid.

In [7]:
configs = [
    dict(name="t_base",   lr0=0.01,  imgsz=640),
    dict(name="t_lr005",  lr0=0.005, imgsz=640),
    dict(name="t_img800", lr0=0.01,  imgsz=800),
    dict(name="t_aug",    lr0=0.01,  imgsz=640, hsv_v=0.5, degrees=5, scale=0.7),
]
rows = []
for c in configs:
    name = c.pop("name")
    YOLO("yolo11n.pt").train(data=DATA_YAML, epochs=15, fraction=0.3, batch=BATCH, seed=SEED,
                             patience=15, device=0, project=SAVE_DIR, name=name, exist_ok=True, **c)
    mv = YOLO(f"{SAVE_DIR}/{name}/weights/best.pt").val(data=DATA_YAML, split="val", imgsz=c["imgsz"],
                                                         batch=BATCH, device=0, plots=False)
    rows.append((name, *map6(mv)))
    print(rows[-1])
print(rows)

Ultralytics 8.4.194 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.3, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=t_base, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, o

## 8. Experiment: merge confusing crack classes
crack + longitudinal + transverse + edge -> `linear_crack` (4 classes). YOLO11n is retrained from scratch on the merged labels. Polygon rows are converted to boxes.

In [6]:
MAP = {0: 0, 1: 1, 2: 2, 3: 2, 4: 2, 5: 3, 6: 2}
NEW_NAMES = ["alligator", "block", "linear_crack", "pothole"]
new_root = "/content/rd_merged"
shutil.rmtree(new_root, ignore_errors=True)

def convert_label(src, dst):
    rows = []
    for line in open(src):
        t = line.split()
        if not t:
            continue
        c = MAP[int(t[0])]
        v = list(map(float, t[1:]))
        if len(v) == 4:
            xc, yc, w, h = v
        else:  # polygon -> bounding box
            xs, ys = v[0::2], v[1::2]
            x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
            xc, yc, w, h = (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1
        rows.append(f"{c} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
    open(dst, "w").write("\n".join(rows))

def build(split, paths):
    os.makedirs(f"{new_root}/{split}/images", exist_ok=True)
    os.makedirs(f"{new_root}/{split}/labels", exist_ok=True)
    out = []
    for p in paths:
        name = os.path.basename(p)
        dst_img = f"{new_root}/{split}/images/{name}"
        shutil.copy(p, dst_img)
        lf = p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
        convert_label(lf, f"{new_root}/{split}/labels/{os.path.splitext(name)[0]}.txt")
        out.append(dst_img)
    open(f"{new_root}/m_{split}.txt", "w").write("\n".join(out))

build("train", train_clean); build("valid", valid_clean); build("test", test_clean)

yaml.dump({"path": new_root, "train": f"{new_root}/m_train.txt", "val": f"{new_root}/m_valid.txt",
           "test": f"{new_root}/m_test.txt", "nc": 4, "names": NEW_NAMES},
          open("/content/data_merged.yaml", "w"))

YOLO("yolo11n.pt").train(data="/content/data_merged.yaml", epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED,
                         patience=PATIENCE, device=0, project=SAVE_DIR, name="yolo11n_merged", exist_ok=True)

best_path = f"{SAVE_DIR}/yolo11n_merged/weights/best.pt"
best = YOLO(best_path)
mv = best.val(data="/content/data_merged.yaml", split="val", imgsz=IMGSZ, batch=BATCH, device=0, plots=False)
mt = best.val(data="/content/data_merged.yaml", split="test", imgsz=IMGSZ, batch=BATCH, device=0, plots=True)

print("RESULT yolo11n_merged")
print("val  mAP50, mAP50-95:", round(mv.box.map50, 3), round(mv.box.map, 3))
print("test mAP50, mAP50-95:", round(mt.box.map50, 3), round(mt.box.map, 3))
print("test precision, recall:", round(mt.box.mp, 3), round(mt.box.mr, 3))
print("inference ms:", round(mt.speed["inference"], 1), "| size MB:", round(os.path.getsize(best_path) / 1e6, 1))

Ultralytics 8.4.201 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_merged.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_merged, nbs=64, nms=None, opset=None, optimize=False, op

## 9. Test samples for trying the app
Exports 3 test images per class as a zip.

In [9]:
import random, shutil, os
from collections import defaultdict

names = {0: "alligator", 1: "block", 2: "crack", 4: "longitudinal", 5: "pothole", 6: "transverse"}
by_class = defaultdict(list)
for p in test_clean:
    lf = p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    for c in {int(l.split()[0]) for l in open(lf) if l.strip()}:
        by_class[c].append(p)

out = "/content/test_samples"
shutil.rmtree(out, ignore_errors=True)
random.seed(1)
for c, n in names.items():
    os.makedirs(f"{out}/{n}", exist_ok=True)
    for p in random.sample(by_class[c], min(3, len(by_class[c]))):
        shutil.copy(p, f"{out}/{n}")

shutil.make_archive("/content/test_samples", "zip", out)
from google.colab import files
files.download("/content/test_samples.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>